In [3]:
print("hello")

hello


In [4]:
%pwd

'c:\\Vinay\\Projects\\Gen_AI\\mech\\Machine_design\\research'

In [5]:
import os
os.chdir("../")

In [6]:
%pwd

'c:\\Vinay\\Projects\\Gen_AI\\mech\\Machine_design'

In [7]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


c:\Users\vinay\anaconda3\envs\mech_bot\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\vinay\anaconda3\envs\mech_bot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="**/*.pdf",   # IMPORTANT FIX
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [9]:
extracted_data = load_pdf_file("data")

In [10]:
extracted_data

[Document(metadata={'source': 'data\\game.pdf', 'page': 0, 'page_label': '129'}, page_content='Team Games\n5\nAny game which provides opportunities to two or more \nplayers working together towards a shared objective is called \na team game. A team game is an activity in which individuals \nare organised in a team to compete with the opposing team, \nin accordance with a set of laws/rules to win. Games like \nBasketball, Cricket, Football, Handball, Hockey, Volleyball, \netc., are some of the classic examples of major team games.\nHowever, over a period of time, the popularity of team \ngames has grown continuously. These games have positively \ninfluenced not just the players, but also their fans, local and \nnational economies. All over the world, the impact of team \ngames can be seen resulting in professional players to live \nout their dreams. Star players have become role models to \nyouth. Young athletes/players develop life skills which are \nfollowed as footsteps of their role

In [11]:
len(extracted_data)

81

In [12]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs : List[Document]) -> List[Document]:

    minimal_docs : List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata ={"source": src}
            )
        )
    return minimal_docs

In [13]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [14]:
minimal_docs

[Document(metadata={'source': 'data\\game.pdf'}, page_content='Team Games\n5\nAny game which provides opportunities to two or more \nplayers working together towards a shared objective is called \na team game. A team game is an activity in which individuals \nare organised in a team to compete with the opposing team, \nin accordance with a set of laws/rules to win. Games like \nBasketball, Cricket, Football, Handball, Hockey, Volleyball, \netc., are some of the classic examples of major team games.\nHowever, over a period of time, the popularity of team \ngames has grown continuously. These games have positively \ninfluenced not just the players, but also their fans, local and \nnational economies. All over the world, the impact of team \ngames can be seen resulting in professional players to live \nout their dreams. Star players have become role models to \nyouth. Young athletes/players develop life skills which are \nfollowed as footsteps of their role models.\nIn this chapter, some 

In [15]:
from typing import List
from langchain_core.documents import Document

def filter_docs_for_rag(docs: List[Document]) -> List[Document]:
    """
    Prepare documents for RAG:
    - Preserve useful metadata
    - Tag document type (text/table/image)
    - Clean content safely
    """

    processed_docs = []

    for doc in docs:
        content = doc.page_content.strip() if doc.page_content else ""

        # Skip empty content
        if not content:
            continue

        metadata = doc.metadata or {}

        # Detect type (default = text)
        doc_type = metadata.get("type", "text")

        # Normalize table content
        if doc_type == "table":
            content = f"[TABLE]\n{content}"

        # Normalize image content (assumes caption/OCR already done)
        elif doc_type == "image":
            content = f"[IMAGE DESCRIPTION]\n{content}"

        # Build enriched metadata
        enriched_metadata = {
            "source": metadata.get("source"),
            "page": metadata.get("page"),
            "type": doc_type,
            "file_name": metadata.get("file_name"),
        }

        processed_docs.append(
            Document(
                page_content=content,
                metadata=enriched_metadata
            )
        )

    return processed_docs

In [16]:
all_docs = filter_docs_for_rag(extracted_data)

In [17]:
all_docs

[Document(metadata={'source': 'data\\game.pdf', 'page': 0, 'type': 'text', 'file_name': None}, page_content='Team Games\n5\nAny game which provides opportunities to two or more \nplayers working together towards a shared objective is called \na team game. A team game is an activity in which individuals \nare organised in a team to compete with the opposing team, \nin accordance with a set of laws/rules to win. Games like \nBasketball, Cricket, Football, Handball, Hockey, Volleyball, \netc., are some of the classic examples of major team games.\nHowever, over a period of time, the popularity of team \ngames has grown continuously. These games have positively \ninfluenced not just the players, but also their fans, local and \nnational economies. All over the world, the impact of team \ngames can be seen resulting in professional players to live \nout their dreams. Star players have become role models to \nyouth. Young athletes/players develop life skills which are \nfollowed as footsteps

In [18]:
# split the documents into smaller chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20,
        length_function = len
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks


In [19]:
text_chunks = text_split(minimal_docs)
print(f"Number of chunks: {len(text_chunks)}")

Number of chunks: 316


In [20]:
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device}
    )
    
    return embeddings

embedding = download_embeddings()

C:\Users\vinay\AppData\Local\Temp\ipykernel_26580\3802109406.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\vinay\anaconda3\envs\mech_bot\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [21]:
vector = embedding.embed_query("I love Cricket")
vector

[0.03202337771654129,
 0.04759501293301582,
 -0.015142214484512806,
 -0.06644179672002792,
 0.04099971801042557,
 0.011611121706664562,
 0.05262907221913338,
 0.01883704587817192,
 0.025178221985697746,
 0.12971465289592743,
 -0.04459289461374283,
 -0.11896835267543793,
 0.017614424228668213,
 0.06525370478630066,
 0.10775531828403473,
 0.0023104327265173197,
 0.019173547625541687,
 -0.08108918368816376,
 -0.017462875694036484,
 -0.08364690095186234,
 -0.10228151082992554,
 0.15134145319461823,
 0.026217704638838768,
 -0.03524390980601311,
 0.00617063045501709,
 -0.008189838379621506,
 0.01214199885725975,
 0.02898779883980751,
 -0.02811475098133087,
 -0.04789691045880318,
 0.032541025429964066,
 0.0990532785654068,
 -0.023475829511880875,
 0.04438658803701401,
 -0.0781792625784874,
 -0.027584755793213844,
 -0.027498921379446983,
 0.04267416521906853,
 0.10368993878364563,
 0.0017614231910556555,
 -0.009152678772807121,
 0.0048359171487390995,
 0.04649142548441887,
 0.00553607475012540

In [22]:
len(vector)

384

In [23]:
#from dotenv import load_dotenv
#import os
#load_dotenv()

In [24]:
#PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
#OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

#os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [25]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env file

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [30]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key = pinecone_api_key)

In [31]:
pc

In [33]:
from pinecone import ServerlessSpec

index_name = "mech-bot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 384,
        metric = "cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
from lanchain_pinecone import PineconeVectorScore

docsearch = PineconeVectorScore.from_documents(
    documents = texts_chunk,
    embedding = embedding,
    index_name = index_name
)

In [ ]:
#load existing index
from lanchain_pinecone import PineconeVectorScore

docsearch = PineconeVectorScore.from_existing_index(
    embedding = embedding,
    index_name = index_name
)

In [ ]:
#add more data to the existing index

dswith = Document(
    page_content = "There are lot of opening for genai",
    metadata = {"source": "internet"}
)


In [ ]:
docsearch.add_documents(documents=[dswith])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k:3"})

In [ ]:
retrieved_docs = retriever.invoke("What is cricket?")

In [ ]:
retrieved_docs

In [ ]:
from langchain_openai import ChatOpenAI
chatModel = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are an assistant for question-answering tasks."
    "Use the follwing pieces of retrieved context for answer."
    "For question if you don't know the answer, you say don't know."
    "keep three sentences, keep the answer concise."
    '\n\n'
    "{context}"
)

prompt = ChatPromptTemplate from_messages(
    [
        ("system", system_prompt),
        ("human","{input})"
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is cricket"})
print(response["answer"])